# Auswertungsskript Yukka-Lab News Daten

In [2]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

### Datenimport CSVs aus Yukka Lab

In [3]:
main_folder = '/Users/maxbreitruck/Library/CloudStorage/OneDrive-UniversitätSt.Gallen/001_IWI/Dissertation/Paper#2/Yukkalab Daten/Analysis_Data'  

company_folders = [os.path.join(main_folder, name) for name in os.listdir(main_folder) if os.path.isdir(os.path.join(main_folder, name))]

print(company_folders)  


['/Users/maxbreitruck/Library/CloudStorage/OneDrive-UniversitätSt.Gallen/001_IWI/Dissertation/Paper#2/Yukkalab Daten/Analysis_Data/company:ubs', '/Users/maxbreitruck/Library/CloudStorage/OneDrive-UniversitätSt.Gallen/001_IWI/Dissertation/Paper#2/Yukkalab Daten/Analysis_Data/company:nestle', '/Users/maxbreitruck/Library/CloudStorage/OneDrive-UniversitätSt.Gallen/001_IWI/Dissertation/Paper#2/Yukkalab Daten/Analysis_Data/company:novartis']


In [4]:
# Function to determine implementation level based on volume
def determine_level(volume, low_threshold, medium_threshold):
    if volume == 0:
        return 0  # No implementation
    elif volume <= low_threshold:
        return 1  # Low
    elif volume <= medium_threshold:
        return 2  # Medium
    else:
        return 3  # High

# Iterate over each company folder
for company_folder in company_folders:
    # Initialize an empty DataFrame for the current company
    company_data = pd.DataFrame()

    # Read all CSV files and concatenate into a single DataFrame
    for file_name in os.listdir(company_folder):
        if file_name.endswith('.csv'):
            file_path = os.path.join(company_folder, file_name)
            temp_df = pd.read_csv(file_path)
            company_data = pd.concat([company_data, temp_df], ignore_index=True)
    
    # Continue with the analysis and plotting for the current company's data
    # Step 1: Extract Pillar Information
    company_data['pillar'] = company_data['practice'].apply(lambda x: x[:3])  # Adjust the slicing based on your actual data

    # Step 2: Loop Over Each Pillar and Then Each Capability Within the Pillar
    for selected_pillar in company_data['pillar'].unique():
        # Focus on the selected pillar
        pillar_data = company_data[company_data['pillar'] == selected_pillar].copy()

        plt.figure(figsize=(12, 8))

        for selected_capability in pillar_data['capability'].unique():
            # Focus on the selected capability within the current pillar
            capability_data = pillar_data[pillar_data['capability'] == selected_capability].copy()

            # Convert 'date' to a monthly period
            capability_data['month'] = capability_data['date'].dt.to_period('M')

            # Aggregate data by month and practice
            monthly_data = capability_data.groupby(['practice', 'month']).agg(
                monthly_volume=('volume', 'sum')
            ).reset_index()

            # Set thresholds for low, medium, high implementation levels based on monthly volume
            low_threshold = monthly_data['monthly_volume'].quantile(0.33)
            medium_threshold = monthly_data['monthly_volume'].quantile(0.66)
            
            # Apply the function to determine implementation level
            monthly_data['implementation_level'] = monthly_data['monthly_volume'].apply(lambda x: determine_level(x, low_threshold, medium_threshold))

            # Ensure non-decreasing implementation level for each practice
            for practice in monthly_data['practice'].unique():
                practice_data = monthly_data[monthly_data['practice'] == practice].copy()
                practice_data['non_decreasing_level'] = practice_data['implementation_level'].cummax()
                
                # Update these values back to the main DataFrame
                monthly_data.loc[monthly_data['practice'] == practice, 'implementation_level'] = practice_data['non_decreasing_level']

            # Calculate Maximum Possible Score
            num_practices = monthly_data['practice'].nunique()
            max_possible_score = num_practices * 3

            # Calculate Actual Score per Month
            monthly_scores = monthly_data.groupby('month')['implementation_level'].sum().reset_index()
            monthly_scores.rename(columns={'implementation_level': 'actual_score'}, inplace=True)

            # Normalize the Actual Score
            monthly_scores['normalized_score'] = (monthly_scores['actual_score'] / max_possible_score) * 100  # multiply by 100 to get a percentage

            # Plot the Normalized Score Over Time for each Capability within the Pillar
            monthly_scores['month'] = monthly_scores['month'].dt.to_timestamp()  # Convert Period to Timestamp for plotting
            sns.lineplot(data=monthly_scores, x='month', y='normalized_score', label=selected_capability, marker='o')

        plt.title(f'Overall Maturity Development Over Time for Pillar: {selected_pillar} in {os.path.basename(company_folder)}')
        plt.xlabel('Month')
        plt.ylabel('Normalized Maturity Score (%)')
        plt.ylim(0, 100)  # Set y-axis from 0 to 100 for percentage
        plt.legend(title='Capability')
        plt.grid(True)  # Adding grid for better readability
        plt.show()

KeyError: 'practice'